# Pruebas A2A - Agente Profesor de Historia

Notebook de pruebas contra el servidor `langgraph dev` local (puerto 2025).
Grafo: `agente_profesor_historia` (`./src/agente_profesor/graph.py:graph`).

## 1) Configuracion e imports

In [1]:
import asyncio
import json
import os
from pathlib import Path

import httpx
from dotenv import find_dotenv, load_dotenv
from langgraph_sdk import get_client

# Cargar credenciales desde agente-profesor-geografia/.env (directorio padre de notebooks)
RUTA_ENV = (Path.cwd().parent / ".env").resolve()
if RUTA_ENV.exists():
    load_dotenv(dotenv_path=RUTA_ENV, override=True)
    ORIGEN_ENV = str(RUTA_ENV)
else:
    ORIGEN_ENV = find_dotenv(usecwd=True)
    if ORIGEN_ENV:
        load_dotenv(dotenv_path=ORIGEN_ENV, override=True)

print("Credenciales cargadas desde .env ->", ORIGEN_ENV or "(no se encontro ningun .env)")

if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError(
        "GOOGLE_API_KEY no esta definida. Defini GOOGLE_API_KEY=tu_clave en "
        "agente-profesor-geografia/.env (ver .env.example) y volve a ejecutar esta celda."
    )

# URLs y grafo tomados del .env cuando existen (A2A_GEOGRAFIA_URL / A2A_GEOGRAFIA_GRAPH)
URL_SERVIDOR = os.environ.get("A2A_GEOGRAFIA_URL", "http://127.0.0.1:2025")
GRAFO = os.environ.get("A2A_GEOGRAFIA_GRAPH", "agente_profesor_geografia")

client = get_client(url=URL_SERVIDOR, api_key=None)


def mostrar(obj):
    """Imprime cualquier objeto de forma legible (JSON indentado si se puede)."""
    try:
        print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))
    except TypeError:
        print(obj)

print("Cliente listo ->", URL_SERVIDOR, "| grafo:", GRAFO)

Credenciales cargadas desde .env -> C:\Users\mwils\Documents\Codigos_python\Codigos\Veolia_codes\AI_agents\VSGPT_instruction_exec\workspace\agente-profesor-geografia\.env
Cliente listo -> http://127.0.0.1:2026 | grafo: agente_profesor_geografia


## 2) Descubrimiento del assistant

In [2]:
assistants = await client.assistants.search()

for a in assistants:
    print(a["assistant_id"], "|", a.get("graph_id"), "|", a.get("name"))

mostrar(assistants)

bed88ad6-7b48-5518-a206-cb2d85c69cda | agente_profesor_geografia | Agente Profesor de Geografía
[
  {
    "assistant_id": "bed88ad6-7b48-5518-a206-cb2d85c69cda",
    "graph_id": "agente_profesor_geografia",
    "config": {},
    "context": {},
    "metadata": {
      "created_by": "system"
    },
    "name": "Agente Profesor de Geografía",
    "created_at": "2026-10-20T22:16:36.877753+00:00",
    "updated_at": "2026-10-20T22:16:57.204789+00:00",
    "version": 2,
    "description": "Especialista en geografía física y humana. Puede: describir climas, relieves y paisajes, explicar cómo el territorio condiciona la actividad humana, y proponer rutas de viaje con su componente geográfico justificado. Habilidades: descripción de territorios, análisis causal territorio-sociedad, diseño de rutas justificadas."
  }
]


In [3]:
# Resolucion automatica del ASSISTANT_ID: se busca por graph_id y, si no hay UUID,
# se usa directamente el nombre del grafo (el servidor lo acepta como identificador).
async def resolver_assistant_id(graph_id: str) -> str:
    try:
        encontrados = await client.assistants.search(graph_id=graph_id)
    except Exception as e:
        print("No se pudo buscar por graph_id:", e)
        encontrados = []
    for a in encontrados or []:
        aid = a.get("assistant_id") if isinstance(a, dict) else None
        if aid:
            return aid
    for a in (assistants or []):
        if isinstance(a, dict) and a.get("graph_id") == graph_id and a.get("assistant_id"):
            return a["assistant_id"]
    return graph_id


ASSISTANT_ID = await resolver_assistant_id(GRAFO)
print("ASSISTANT_ID =", ASSISTANT_ID)

ASSISTANT_ID = bed88ad6-7b48-5518-a206-cb2d85c69cda


## 3) Consulta directa al agente (runs.wait)

In [4]:
def crear_entrada(pregunta: str) -> dict:
    """Arma el input del grafo a partir de una pregunta de texto."""
    return {"messages": [{"role": "human", "content": pregunta}]}


def extraer_texto(resultado) -> str:
    """Devuelve el contenido del ultimo mensaje del resultado de un run."""
    mensajes = (resultado or {}).get("messages", [])
    if not mensajes:
        return ""
    ultimo = mensajes[-1]
    contenido = ultimo.get("content") if isinstance(ultimo, dict) else getattr(ultimo, "content", "")
    if isinstance(contenido, list):
        partes = []
        for p in contenido:
            if isinstance(p, dict) and p.get("type") == "text":
                partes.append(p.get("text", ""))
            elif isinstance(p, str):
                partes.append(p)
        return "\n".join(partes)
    return contenido or ""


resultado = await client.runs.wait(
    None,
    ASSISTANT_ID,
    input=crear_entrada("Explicame el clima y el relieve del valle del Nilo."),
)

print(extraer_texto(resultado))

Para explicar con precisión el clima y el relieve del **valle del Nilo**, debemos situarlo primero en su gran marco regional: el nordeste de África, atravesando desde los Grandes Lagos y las tierras altas de Etiopía hasta su desembocadura en el mar Mediterráneo.

A continuación, detallo el marco general y los dos componentes solicitados:

---

### 1. El Relieve: Un corredor fluvial en medio del desierto

El relieve del valle del Nilo se caracteriza por un contraste absoluto y abrupto entre la estrecha franja fértil fluvial y las mesetas desérticas circundantes.

*   **El cauce y el valle fluvial:** El Nilo discurre por una depresión tectónica y erosiva. En su tramo egipcio (el más clásico como "valle"), el valle es una planicie aluvial muy estrecha, con un ancho que oscala entre los 2 y los 20 kilómetros, limitada por escarpes rocosos. 
*   **Los altiplanos y desiertos circundantes:** 
    *   Al oeste se extiende el **Desierto Líbico** (parte del Sáhara), un vasto mar de arena, dunas 

## 4) Prueba de memoria (thread con dos preguntas encadenadas)

In [5]:
thread = await client.threads.create()
thread_id = thread["thread_id"]
print("thread_id =", thread_id)

r1 = await client.runs.wait(
    thread_id,
    ASSISTANT_ID,
    input=crear_entrada("Que concepto geografico explica que la agricultura se concentrara en las cuencas de los rios?"),
)
print("--- Respuesta 1 ---")
print(extraer_texto(r1))

r2 = await client.runs.wait(
    thread_id,
    ASSISTANT_ID,
    input=crear_entrada("Y como se aplica ese mismo concepto al caso del valle del Indo?"),
)
print("--- Respuesta 2 (deberia recordar el contexto) ---")
print(extraer_texto(r2))

thread_id = 01a15b65-19d1-76a7-b9d4-0cd39679e1a1


--- Respuesta 1 ---
¡Hola! Como profesor de geografía, con gusto respondo a tu consulta.

El concepto geográfico fundamental que explica la concentración histórica de la agricultura en las cuencas de los ríos es el **determinismo ambiental** (en su formulación clásica e histórica, donde el medio físico condiciona fuertemente las posibilidades humanas) y, de forma más precisa desde la perspectiva de la geografía humana y ambiental actual, el concepto de **ventaja locacional** y **disponibilidad de recursos hídricos y edáficos**. 

Para desglosarlo con el rigor metodológico que corresponde, veamos el marco general y el detalle:

### 1. Marco general: El nicho ecológico de las primeras civilizaciones
Las grandes cuencas fluviales del mundo (como el Tigris y el Éufrates en Mesopotamia, el Nilo en Egipto, el Indo en el subcontinente indio o el Yangtsé y el Amarillo en China) fueron los escenarios donde surgió la agricultura intensiva y las primeras sociedades urbanas (la llamada *Revolución

--- Respuesta 2 (deberia recordar el contexto) ---
Para explicar cómo se aplica el concepto de concentración agrícola en cuencas fluviales al **caso específico del valle del Indo**, debemos situarnos en el noroeste del subcontinente indio (actuales Pakistán y noroccidente de la India), en plena **cuenca del río Indo y sus cinco afluentes** (región del Punyab). 

Dado que este caso específico no figura en mi cuaderno de apuntes, la siguiente explicación proviene de mi **conocimiento general en geografía histórica y física**:

### 1. Marco geográfico y climatológico
El valle del Indo se ubica en una región de transición climática árida y semiárida, circundada por sistemas montañosos formidables (el Hindu Kush y el Himalaya) y el desierto de Thar hacia el este. En un entorno donde las precipitaciones pluviales son escasas e irregulares, **el agua de lluvia por sí sola es insuficiente para sostener una agricultura intensiva**. 

### 2. El factor hidrológico y edafológico (Ventaja locaciona

## 5) Grafo coordinador minimo que delega en el agente remoto

In [6]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class EstadoCoordinador(TypedDict):
    pregunta: str
    respuesta: str


async def nodo_profesor_historia(state: EstadoCoordinador) -> EstadoCoordinador:
    resultado = await client.runs.wait(
        None,
        ASSISTANT_ID,
        input=crear_entrada(state["pregunta"]),
    )
    return {"respuesta": extraer_texto(resultado)}


builder = StateGraph(EstadoCoordinador)
builder.add_node("profesor_historia", nodo_profesor_historia)
builder.add_edge(START, "profesor_historia")
builder.add_edge("profesor_historia", END)
coordinador = builder.compile()

print(coordinador.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	profesor_historia(profesor_historia)
	__end__([<p>__end__</p>]):::last
	__start__ --> profesor_historia;
	profesor_historia --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [7]:
salida = await coordinador.ainvoke(
    {"pregunta": "Describime el relieve y los ambientes de la meseta tibetana.", "respuesta": ""}
)
print(salida["respuesta"])

Para abordar el estudio de la **Meseta Tibetana** (conocida geográficamente como la meseta de Tíbet o *Qinghai-Xizang*), es necesario establecer primero su marco espacial y físico general, ya que nos encontramos ante una de las regiones geográficas más singulares y extremas del planeta.

---

### 1. Marco General y Ubicación
* **Localización:** Situada en Asia central y oriental, abarca principalmente la Región Autónoma del Tíbet y la provincia de Qinghai en China, extendiéndose parcialmente hacia Ladakh (India).
* **Dimensiones:** Ocupa una superficie aproximada de 2,5 millones de kilómetros cuadrados (equivalente a casi cuatro veces el tamaño de Texas o España).
* **Altitud media:** Supera los **4.500 metros sobre el nivel del mar**, razón por la cual es universalmente denominada el *"Techo del Mundo"* o el *"Tercer Polo"*.

*Aclaración:* Este concepto y sus datos generales provienen de mi conocimiento geográfico general, ya que no figura registrado en mi cuaderno de apuntes particul

## 6) Agent2Agent (a2a-sdk): Agent Card y cliente

In [8]:
from google.protobuf.json_format import MessageToDict
from a2a.client import A2ACardResolver, ClientConfig, create_client

URL_A2A = os.environ.get("A2A_GEOGRAFIA_URL", "http://127.0.0.1:2025")

async with httpx.AsyncClient(timeout=60.0) as http_client:
    resolver = A2ACardResolver(httpx_client=http_client, base_url=URL_A2A)
    agent_card = await resolver.get_agent_card(
        http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
    )
    mostrar(MessageToDict(agent_card))


{
  "name": "Agente Profesor de Geografía",
  "description": "Especialista en geografía física y humana. Puede: describir climas, relieves y paisajes, explicar cómo el territorio condiciona la actividad humana, y proponer rutas de viaje con su componente geográfico justificado. Habilidades: descripción de territorios, análisis causal territorio-sociedad, diseño de rutas justificadas.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2026/a2a/bed88ad6-7b48-5518-a206-cb2d85c69cda",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "0.16.2",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "params": {
          "default": "context",
          "header": "LangGraph-A2A-History-Scope",
          "values": [
            "context",

In [9]:
from a2a.types import Role, SendMessageRequest, TaskState
from a2a.helpers import new_text_message, get_message_text, get_artifact_text


async def crear_cliente_a2a(card, http_client):
    """Fuerza la version 0.3.0 (compatible con a2a-sdk 1.2.x) y crea el cliente."""
    for interfaz in card.supported_interfaces:
        if interfaz.protocol_binding == "JSONRPC":
            interfaz.protocol_version = "0.3.0"
    config = ClientConfig(
        httpx_client=http_client,
        streaming=False,
        supported_protocol_bindings=["JSONRPC"],
    )
    return await create_client(card, client_config=config)


async with httpx.AsyncClient(timeout=60.0) as http_client:
    resolver = A2ACardResolver(httpx_client=http_client, base_url=URL_A2A)
    agent_card = await resolver.get_agent_card(
        http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
    )
    a2a_client = await crear_cliente_a2a(agent_card, http_client)

    request = SendMessageRequest(
        message=new_text_message("Que diferencia hay entre clima y tiempo, y por que importa?", role=Role.ROLE_USER)
    )

    async for evento in a2a_client.send_message(request):
        if evento.HasField("message"):
            print(get_message_text(evento.message))
        elif evento.HasField("task"):
            tarea = evento.task
            if tarea.status.state != TaskState.TASK_STATE_COMPLETED:
                estado = TaskState.Name(tarea.status.state)
                raise RuntimeError(f"Tarea A2A {tarea.id} no finalizada: {estado}")
            for artifact in tarea.artifacts:
                print(get_artifact_text(artifact))
            if not tarea.artifacts and tarea.status.HasField("message"):
                print(get_message_text(tarea.status.message))


¡Hola! Como profesor de geografía, esta es una de las primeras distinciones fundamentales que debemos clarificar, ya que a menudo se usan como sinónimos en el lenguaje cotidiano, pero representan realidades espaciales y temporales muy diferentes.

A continuación, te presento el marco general y el detalle de ambos conceptos, y posteriormente integraré lo que recogen mis notas personales sobre el tema.

---

### 1. Marco General y Definiciones

*   **El tiempo atmosférico (o tiempo meteorológico):** Es el estado de la atmósfera en un **lugar específico y en un momento determinado** (minutos, horas o pocos días). Es altamente dinámico y cambiante.
    *   *Ejemplo:* "Hoy en Madrid está lloviendo y la temperatura es de 14 °C, pero ayer hizo sol".
*   **El clima:** Es el patrón promedio de las condiciones atmosféricas en una **región amplia**, calculado a lo largo de un **período prolongado** (habitualmente series estadísticas de 30 años, según la Organización Meteorológica Mundial). No des

## 7) Ejemplo completo: router/supervisor + nodo A2A remoto

In [10]:
from typing import Literal, Optional, TypedDict
from uuid import uuid4

from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph

from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Role, SendMessageRequest, TaskState
from a2a.helpers import new_text_message, get_message_text, get_artifact_text


# Catalogo de agentes remotos disponibles (A2A), tomado del .env cuando existe
CATALOGO_URLS = {
    GRAFO: URL_A2A,
}


async def descubrir_catalogo() -> dict:
    catalogo = {}
    async with httpx.AsyncClient(timeout=60.0) as http_client:
        for nombre, url in CATALOGO_URLS.items():
            try:
                resolver = A2ACardResolver(httpx_client=http_client, base_url=url)
                card = await resolver.get_agent_card(
                    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
                )
                catalogo[nombre] = {
                    "url": url,
                    "descripcion": card.description,
                    "skills": [s.name for s in (card.skills or [])],
                }
            except Exception as e:  # servidor no disponible
                catalogo[nombre] = {"url": url, "error": str(e)}
    return catalogo


catalogo = await descubrir_catalogo()
mostrar(catalogo)


{
  "agente_profesor_geografia": {
    "url": "http://127.0.0.1:2026",
    "descripcion": "Especialista en geografía física y humana. Puede: describir climas, relieves y paisajes, explicar cómo el territorio condiciona la actividad humana, y proponer rutas de viaje con su componente geográfico justificado. Habilidades: descripción de territorios, análisis causal territorio-sociedad, diseño de rutas justificadas.",
    "skills": [
      "Agente Profesor de Geografía Capabilities"
    ]
  }
}


In [11]:
class DecisionRouter(BaseModel):
    destino: Literal["agente_remoto", "responder_directo"] = Field(
        description="A donde enviar la consulta del usuario"
    )
    agente: Optional[str] = Field(
        default=None, description="Nombre del agente remoto del catalogo"
    )
    motivo: str = Field(description="Justificacion breve de la decision")


MODELO = os.environ.get("MODELO", "gemini-3.5-flash-lite")
llm = ChatGoogleGenerativeAI(model=MODELO, temperature=0)
llm_router = llm.with_structured_output(DecisionRouter)


class EstadoSupervisor(TypedDict):
    pregunta: str
    agente: Optional[str]
    motivo: str
    respuesta: str


async def nodo_router(state: EstadoSupervisor) -> dict:
    prompt = (
        "Sos un supervisor que decide si delegar una consulta a un agente remoto.\n"
        f"Catalogo de agentes disponibles: {json.dumps(catalogo, ensure_ascii=False, default=str)}\n"
        "Si la consulta es de historia, delegar en 'agente_profesor_geografia'.\n"
        f"Consulta del usuario: {state['pregunta']}"
    )
    decision = await llm_router.ainvoke(prompt)
    return {
        "agente": decision.agente or GRAFO,
        "motivo": decision.motivo,
        "respuesta": "",
    }


async def nodo_agente_remoto(state: EstadoSupervisor) -> dict:
    nombre = state.get("agente") or GRAFO
    url = CATALOGO_URLS.get(nombre, URL_A2A)

    async with httpx.AsyncClient(timeout=120.0) as http_client:
        resolver = A2ACardResolver(httpx_client=http_client, base_url=url)
        card = await resolver.get_agent_card(
            http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
        )
        cliente = await crear_cliente_a2a(card, http_client)

        request = SendMessageRequest(
            message=new_text_message(state["pregunta"], role=Role.ROLE_USER)
        )

        textos = []
        async for evento in cliente.send_message(request):
            if evento.HasField("message"):
                textos.append(get_message_text(evento.message))
            elif evento.HasField("task"):
                tarea = evento.task
                if tarea.status.state != TaskState.TASK_STATE_COMPLETED:
                    estado = TaskState.Name(tarea.status.state)
                    raise RuntimeError(f"Tarea A2A {tarea.id} no finalizada: {estado}")
                textos.extend(get_artifact_text(a) for a in tarea.artifacts)
                if not textos and tarea.status.HasField("message"):
                    textos.append(get_message_text(tarea.status.message))

        respuesta = "\n".join(filter(None, textos))
        if not respuesta:
            raise RuntimeError("El agente remoto no devolvio texto.")
        return {"respuesta": respuesta}


async def nodo_responder_directo(state: EstadoSupervisor) -> dict:
    r = await llm.ainvoke(state["pregunta"])
    return {"respuesta": r.content}


sb = StateGraph(EstadoSupervisor)
sb.add_node("router", nodo_router)
sb.add_node("agente_remoto", nodo_agente_remoto)
sb.add_node("responder_directo", nodo_responder_directo)
sb.add_edge(START, "router")
sb.add_conditional_edges(
    "router",
    lambda s: "agente_remoto" if s.get("agente") else "responder_directo",
    {"agente_remoto": "agente_remoto", "responder_directo": "responder_directo"},
)
sb.add_edge("agente_remoto", END)
sb.add_edge("responder_directo", END)
supervisor = sb.compile()

print(supervisor.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	agente_remoto(agente_remoto)
	responder_directo(responder_directo)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	router -.-> agente_remoto;
	router -.-> responder_directo;
	agente_remoto --> __end__;
	responder_directo --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [12]:
salida = await supervisor.ainvoke(
    {
        "pregunta": "Explicame las consecuencias de la Primera Guerra Mundial.",
        "agente": None,
        "motivo": "",
        "respuesta": "",
    }
)
print("Motivo:", salida["motivo"])
print("Agente:", salida["agente"])
print()
print(salida["respuesta"])

C:\Users\mwils\Documents\Codigos_python\Codigos\Veolia_codes\AI_agents\VSGPT_instruction_exec\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Motivo: La consulta es sobre la Primera Guerra Mundial y, segun las reglas explicitas, las consultas de historia deben delegarse al agente_profesor_geografia.
Agente: agente_profesor_geografia

Como profesor de geografía, debo señalar en primer lugar que **la historia, la geopolítica y los tratados internacionales estrictos no son mi especialidad principal**; para un análisis detallado de los acuerdos diplomáticos y las decisiones políticas de la época, correspondería derivar al historiador contemporáneo o al especialista en relaciones internacionales. 

Sin embargo, **la Primera Guerra Mundial (1914-1918) transformó radicalmente el mapa físico, político y humano del planeta**, un ámbito que sí compete de lleno a la geografía humana y política. A continuación, expongo las principales consecuencias territoriales, demográficas y socioeconómicas de aquel conflicto:

### 1. Transformación territorial y colapso de los grandes imperios
El mapa de Europa y Oriente Medio se rediseñó por comple